In [35]:
import torch
from torch.export import export
from transformers import AutoConfig, AutoModel
import torch.nn as nn

import timm
from torchvision import models



In [ ]:
model_name = 'gpt2'
config = AutoConfig.from_pretrained(model_name)
config.use_cache = False  # DynamicCache hatasını engellemek için
model = AutoModel.from_config(config).eval()

example_args = (torch.randint(0, config.vocab_size, (1, 128)),)

exported_program = export(model, args=example_args, kwargs={"return_dict": False})
graph_module = exported_program.graph_module

dag_features = {}

def get_tensor_size(node_output):
    """Tensörün toplam byte boyutunu hesaplar (float32 varsayımıyla)"""
    if hasattr(node_output, 'shape'):
        num_elements = 1
        for dim in node_output.shape:
            num_elements *= dim
        return num_elements * 4  # 4 bytes per float32
    return 0

In [ ]:
def calculate_flops(node):
    target = node.target
    out_val = node.meta.get('val')
    
    # 1. SENARYO: Matris Çarpımları (Linear, Matmul, addmm)
    if target in [torch.ops.aten.mm.default, torch.ops.aten.addmm.default, torch.ops.aten.bmm.default]:
        try:
            # addmm(bias, mat1, mat2) -> indis 1 ve 2 | mm(mat1, mat2) -> 0 ve 1
            idx1, idx2 = (1, 2) if target == torch.ops.aten.addmm.default else (0, 1)
            arg1 = node.args[idx1].meta.get('val')
            arg2 = node.args[idx2].meta.get('val')
            
            if arg1 is not None and arg2 is not None:
                s1, s2 = list(arg1.shape), list(arg2.shape)
                # Batch matmul (bmm) ise en baştaki batch boyutunu sona sakla
                batch_dim = s1[0] if target == torch.ops.aten.bmm.default else 1
                
                # Boyutları temizle (Son iki boyuta odaklan: m, n, k)
                m = s1[-2] if len(s1) >= 2 else 1
                n = s1[-1]
                k = s2[-1] if len(s2) >= 2 else s2[0]
                
                return 2 * batch_dim * m * n * k
        except: return 0

    # 2. SENARYO: Evrişimli Katmanlar (CNN'ler için hayati)
    elif target == torch.ops.aten.convolution.default:
        try:
            # args[0]: input, args[1]: weight
            input_val = node.args[0].meta.get('val')
            weight_val = node.args[1].meta.get('val')
            if input_val is not None and out_val is not None:
                # out_val shape: [B, C_out, H_out, W_out]
                # weight shape: [C_out, C_in, K_h, K_w]
                output_elements = out_val.numel()
                kernel_ops = weight_val.shape[1] * weight_val.shape[2] * weight_val.shape[3]
                return 2 * output_elements * kernel_ops
        except: return 0

    # 3. SENARYO: Scaled Dot Product Attention (Transformer'ların kalbi)
    elif "scaled_dot_product_attention" in str(target):
        try:
            # args[0]: query (B, H, L, D)
            q = node.args[0].meta.get('val')
            if q is not None:
                b, h, l, d = q.shape
                # Q*K^T + Softmax*V maliyeti (Basitleştirilmiş standart formül)
                return 2 * b * h * l * l * d
        except: return 0

    # 4. SENARYO: Element-wise Operasyonlar (ReLU, Add, Mul, LayerNorm)
    # Bunlar genelde O(N) maliyetlidir ,tek başlarına küçük ama toplamda etkilidirler.
    elif out_val is not None:
        # Eğer yukarıdaki ağır işlemlere girmediyse ama bir çıktı üretiyorsa,
        # en azından tensörün eleman sayısı kadar işlem yapilmistir (aktivasyonlar gibi).
        return out_val.numel()

    return 0

In [42]:

# --- YENİ YARDIMCI FONKSİYONLAR (Listenin içindeki tüm tensörleri toplar) ---
def get_safe_numel(val):
    if hasattr(val, 'numel'): # Eğer tek bir tensörse
        return val.numel()
    elif isinstance(val, (list, tuple)): # Eğer tensör listesiyse (split vb.)
        return sum(get_safe_numel(v) for v in val)
    return 0

def get_safe_size(val):
    return get_safe_numel(val) * 4

print(f"{'Node Name':<30} | {'FLOPs':<12} | {'Mem_Byte':<12} | {'Trans_Byte'}")
print("-" * 80)

for node in graph_module.graph.nodes:
    if node.op == 'placeholder': continue 
    
    flops = 0
    mem_byte = 0
    transfer_byte = 0
    out_val = node.meta.get('val')

    # --- FEATURE 1: Memory Byte (Ağırlık Yükü) ---
    if node.op in ['call_module', 'call_function']:
        for arg in node.args:
            if hasattr(arg, 'meta') and 'val' in arg.meta:
                mem_byte += get_safe_size(arg.meta['val'])

    # --- FEATURE 2: Transfer Byte (İletişim/Data Volume) ---
    if out_val is not None:
        transfer_byte = get_safe_size(out_val)

    # --- FEATURE 3: FLOPs ---
    target = node.target
    
    # 3a. Matris Çarpımları
    if target in [torch.ops.aten.mm.default, torch.ops.aten.addmm.default, torch.ops.aten.bmm.default]:
        try:
            idx1, idx2 = (1, 2) if target == torch.ops.aten.addmm.default else (0, 1)
            arg1 = node.args[idx1].meta.get('val')
            arg2 = node.args[idx2].meta.get('val')
            
            if arg1 is not None and arg2 is not None:
                s1, s2 = list(arg1.shape), list(arg2.shape)
                batch_dim = s1[0] if target == torch.ops.aten.bmm.default else 1
                m = s1[-2] if len(s1) >= 2 else 1
                n = s1[-1]
                k = s2[-1] if len(s2) >= 2 else s2[0]
                flops = 2 * batch_dim * m * n * k
        except: flops = 0

    # 3b. Convolution
    elif target == torch.ops.aten.convolution.default:
        try:
            input_val = node.args[0].meta.get('val')
            weight_val = node.args[1].meta.get('val')
            if out_val is not None and weight_val is not None:
                kernel_ops = weight_val.shape[1] * weight_val.shape[2] * weight_val.shape[3]
                flops = 2 * get_safe_numel(out_val) * kernel_ops
        except: flops = 0

    # 3c. Attention
    elif "scaled_dot_product_attention" in str(target):
        try:
            q = node.args[0].meta.get('val')
            if q is not None:
                b, h, l, d = q.shape
                flops = 2 * b * h * l * l * d
        except: flops = 0
        
    # 3d. Fallback (Hata veren yer burasıydı, artık safe_numel kullanıyor)
    elif out_val is not None and flops == 0:
        flops = get_safe_numel(out_val)

    # --- VERİLERİ KAYDET ---
    dag_features[node.name] = {
        "flops": float(flops),
        "mem_byte": float(mem_byte),
        "transfer_byte": float(transfer_byte),
        "parents": [p.name for p in node.all_input_nodes]
    }

    if flops > 1000 or transfer_byte > 1000:
        print(f"{node.name[:30]:<30} | {flops:<12.0f} | {mem_byte:<12.0f} | {transfer_byte}")

Node Name                      | FLOPs        | Mem_Byte     | Trans_Byte
--------------------------------------------------------------------------------
embedding                      | 98304        | 154390016    | 393216
embedding_1                    | 98304        | 3146240      | 393216
to                             | 98304        | 393216       | 393216
add                            | 98304        | 786432       | 393216
le                             | 16384        | 1024         | 65536
to_1                           | 16384        | 65536        | 65536
and_1                          | 16384        | 65540        | 65536
eq                             | 16384        | 1024         | 65536
to_2                           | 16384        | 65536        | 65536
and_2                          | 16384        | 131072       | 65536
expand                         | 16384        | 65536        | 65536
dropout                        | 98304        | 393216       | 393216
layer_norm  

In [2]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK A — SETUP: imports, core helpers, feature extractor, safe exporter
# ══════════════════════════════════════════════════════════════════════
import gc, json, time, traceback, warnings
warnings.filterwarnings('ignore')

import torch
from torch.export import export
import timm
from torchvision import models as tv_models
from transformers import AutoConfig, AutoModel

# ──────────────────────────────────────────────────────────────────────
# 1. Low-level helpers (tensor / list-of-tensors safe)
# ──────────────────────────────────────────────────────────────────────
def get_safe_numel(val):
    """Count elements in a tensor OR a list/tuple of tensors (split/chunk outputs)."""
    if val is None:              return 0
    if hasattr(val, 'numel'):    return val.numel()
    if isinstance(val, (list, tuple)): return sum(get_safe_numel(v) for v in val)
    return 0

def get_safe_size(val):
    """Byte size assuming float32 (4 B/element)."""
    return get_safe_numel(val) * 4


# ──────────────────────────────────────────────────────────────────────
# 2. Per-node DAG feature extractor
# ──────────────────────────────────────────────────────────────────────
def extract_dag_features(graph_module):
    """
    Walk every node of a torch.export GraphModule and compute:
      flops         – multiply-accumulate count (×2 for FMA)
      mem_byte      – bytes read from predecessor tensors  (weight-load proxy)
      transfer_byte – bytes produced by this node          (data-volume proxy)
      parents       – list of predecessor node names
    Edge cases handled:
      • split/chunk outputs  → list-of-tensors safe via get_safe_numel
      • LayerNorm / BatchNorm → fallback O(N) flop estimate
      • 3-D / grouped convolutions → full kernel shape product
      • disentangled / relative-position attention → einsum path
    """
    dag = {}
    for node in graph_module.graph.nodes:
        flops, mem_byte, transfer_byte = 0, 0, 0
        out_val = node.meta.get('val')
        tgt     = node.target

        # Placeholders = network inputs; zero compute cost
        if node.op == 'placeholder':
            dag[node.name] = {
                "flops": 0.0, "mem_byte": 0.0,
                "transfer_byte": float(get_safe_size(out_val)),
                "parents": []
            }
            continue

        # ── Feature 1 : memory byte (weight-read proxy) ─────────────
        if node.op in ('call_module', 'call_function'):
            for arg in node.args:
                if hasattr(arg, 'meta') and 'val' in arg.meta:
                    mem_byte += get_safe_size(arg.meta['val'])

        # ── Feature 2 : transfer byte (output data volume) ───────────
        if out_val is not None:
            transfer_byte = get_safe_size(out_val)

        # ── Feature 3 : FLOPs ────────────────────────────────────────

        # 3a. Dense matrix multiplications  (Linear → addmm / mm / bmm)
        if tgt in (torch.ops.aten.mm.default,
                   torch.ops.aten.addmm.default,
                   torch.ops.aten.bmm.default):
            try:
                i1, i2 = (1, 2) if tgt == torch.ops.aten.addmm.default else (0, 1)
                a1 = node.args[i1].meta.get('val')
                a2 = node.args[i2].meta.get('val')
                if a1 is not None and a2 is not None:
                    s1, s2 = list(a1.shape), list(a2.shape)
                    B  = s1[0] if tgt == torch.ops.aten.bmm.default else 1
                    M  = s1[-2] if len(s1) >= 2 else 1
                    N  = s1[-1]
                    K  = s2[-1] if len(s2) >= 2 else s2[0]
                    flops = 2 * B * M * N * K
            except Exception: pass

        # 3b. Convolutions  (weight shape: [C_out, C_in/g, *kernel_dims])
        elif tgt == torch.ops.aten.convolution.default:
            try:
                w = node.args[1].meta.get('val')
                if w is not None and out_val is not None:
                    k_ops = 1
                    for d in list(w.shape)[1:]:   # C_in/g × K_h × K_w (× K_d for 3-D)
                        k_ops *= d
                    flops = 2 * get_safe_numel(out_val) * k_ops
            except Exception: pass

        # 3c. Scaled Dot-Product Attention  (Q: B × H × L × D)
        elif 'scaled_dot_product_attention' in str(tgt):
            try:
                q = node.args[0].meta.get('val')
                if q is not None and len(q.shape) >= 4:
                    B, H, L, D = q.shape[0], q.shape[1], q.shape[2], q.shape[3]
                    flops = 2 * B * H * L * L * D   # QK^T  +  softmax·V
            except Exception: pass

        # 3d. Einsum  (disentangled / relative-position attention in DeBERTa etc.)
        elif tgt == torch.ops.aten.einsum.default:
            try:
                if out_val is not None:
                    flops = get_safe_numel(out_val) * 2
            except Exception: pass

        # 3e. Element-wise fallback  (LayerNorm, GELU, Add, Mul, ReLU …)
        #     Cost ≈ O(N)  —  ensures LayerNorm/BN are never silently zeroed
        elif out_val is not None:
            flops = get_safe_numel(out_val)

        dag[node.name] = {
            "flops":         float(flops),
            "mem_byte":      float(mem_byte),
            "transfer_byte": float(transfer_byte),
            "parents":       [p.name for p in node.all_input_nodes]
        }
    return dag


# ──────────────────────────────────────────────────────────────────────
# 3. Safe exporter  (strict mode → non-strict fallback)
# ──────────────────────────────────────────────────────────────────────
def safe_export(model, args, kwargs):
    """
    Returns a graph_module.
    First tries strict=True (full IR-level tracing).
    Falls back to strict=False (torch.fx symbolic trace) on failure.
    Raises RuntimeError if both modes fail.
    """
    try:
        ep = export(model, args=args, kwargs=kwargs, strict=True)
        return ep.graph_module, "strict"
    except Exception as e1:
        try:
            ep = export(model, args=args, kwargs=kwargs, strict=False)
            return ep.graph_module, "non-strict"
        except Exception as e2:
            raise RuntimeError(
                f"strict=True : {str(e1)[:120]}\n"
                f"strict=False: {str(e2)[:120]}"
            ) from e2


# ──────────────────────────────────────────────────────────────────────
# 4. Model-specific input generators
# ──────────────────────────────────────────────────────────────────────
def get_vision_input(model_name):
    """Return (args, kwargs) for vision models."""
    name_lo = model_name.lower()
    if any(k in name_lo for k in ('inception_v3', 'inception_v4', 'inceptionresnet')):
        return (torch.randn(1, 3, 299, 299),), {}
    if any(k in name_lo for k in ('swin', 'maxvit')):
        # Some Swin / MaxViT require multiples of window_size; 256 is safest
        return (torch.randn(1, 3, 256, 256),), {}
    return (torch.randn(1, 3, 224, 224),), {}


def get_hf_input(config):
    """
    Return (args, kwargs) and mutate config in-place for safe export.
    Handles encoder and decoder models; special seq-len for sparse-attention models.
    """
    if hasattr(config, 'use_cache'):
        config.use_cache = False

    model_type = getattr(config, 'model_type', '').lower()
    vocab_size  = getattr(config, 'vocab_size', 30522)
    seq_len     = 128

    # Sparse-attention models need larger / aligned sequence lengths
    if model_type == 'longformer':
        window = getattr(config, 'attention_window', [512])
        window = window[0] if isinstance(window, list) else window
        seq_len = max(window, 128)                     # at least 1 full window
    elif model_type in ('big_bird', 'bigbird_roberta'):
        seq_len = 512                                  # needs ≥ block_size × 3
    elif model_type == 'reformer':
        seq_len = 128                                  # LSH bucket size power-of-2
    elif model_type == 'canine':
        vocab_size = 65536                             # Unicode code points

    vocab_size = min(vocab_size, 250002)               # safety clamp
    input_ids  = torch.randint(0, max(vocab_size, 2), (1, seq_len))
    args       = (input_ids,)
    kwargs     = {"return_dict": False}
    return args, kwargs


print(f"Setup complete — PyTorch {torch.__version__} | timm {timm.__version__}")


Setup complete — PyTorch 2.10.0+cpu | timm 1.0.26


In [3]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK B — MODEL CANDIDATE LISTS  (≤ ~350 M params each)
#   ▸ TIMM_CANDIDATES     : ~110 vision models
#   ▸ TV_CANDIDATES       : ~50  torchvision models
#   ▸ HF_CANDIDATES       : ~75  HuggingFace transformer models
# Total ≈ 235 candidates → target 200 successes
# ══════════════════════════════════════════════════════════════════════

# ── TIMM ──────────────────────────────────────────────────────────────
TIMM_CANDIDATES = [
    # ResNets
    "resnet18", "resnet34", "resnet50", "resnet101", "resnet152",
    "wide_resnet50_2", "wide_resnet101_2",
    "resnext50_32x4d", "resnext101_32x8d",
    # EfficientNets
    "efficientnet_b0", "efficientnet_b1", "efficientnet_b2",
    "efficientnet_b3", "efficientnet_b4", "efficientnet_b5",
    "efficientnet_lite0", "tf_efficientnet_b0",
    # ViT  (≤ Base; Large ~307 M is too heavy on CPU)
    "vit_tiny_patch16_224", "vit_small_patch16_224",
    "vit_base_patch16_224", "vit_base_patch32_224",
    "vit_small_patch32_224",
    # DeiT
    "deit_tiny_patch16_224", "deit_small_patch16_224",
    "deit_base_patch16_224",
    "deit3_small_patch16_224", "deit3_base_patch16_224",
    # Swin Transformer (≤ Base; Large skipped)
    "swin_tiny_patch4_window7_224", "swin_small_patch4_window7_224",
    "swin_base_patch4_window7_224",
    # Swin V2
    "swinv2_tiny_window8_256", "swinv2_small_window8_256",
    # ConvNeXt
    "convnext_tiny", "convnext_small", "convnext_base",
    # ConvNeXt V2
    "convnextv2_tiny", "convnextv2_small", "convnextv2_base",
    # MobileNets
    "mobilenetv2_100", "mobilenetv2_110d", "mobilenetv2_140",
    "mobilenetv3_small_050", "mobilenetv3_small_100",
    "mobilenetv3_large_100", "mobilenetv3_large_075",
    # DenseNets
    "densenet121", "densenet161", "densenet169", "densenet201",
    # RegNets
    "regnetx_002", "regnetx_004", "regnetx_008", "regnetx_016",
    "regnety_002", "regnety_004", "regnety_008", "regnety_016",
    # MLP-Mixer (≤ Base; Large skipped)
    "mixer_s16_224", "mixer_b16_224",
    # ResMLP
    "resmlp_12_224", "resmlp_24_224", "resmlp_36_224",
    # CaiT
    "cait_xxs24_224", "cait_s24_224",
    # LeViT
    "levit_128s", "levit_128", "levit_192", "levit_256", "levit_384",
    # NFNets
    "nfnet_f0", "nfnet_f1",
    "eca_nfnet_l0", "eca_nfnet_l1",
    # SKNet
    "skresnet18", "skresnet34",
    # Res2Net
    "res2net50_26w_4s",
    # Inception (inception_v4 / inceptionresnetv2 are large — keep only v3)
    "inception_v3",
    # Twins
    "twins_svt_small", "twins_svt_base",
    "twins_pcpvt_small", "twins_pcpvt_base",
    # PVT
    "pvt_v2_b0", "pvt_v2_b1", "pvt_v2_b2",
    # ConvMixer
    "convmixer_768_32", "convmixer_1024_20",
    # MaxViT
    "maxvit_tiny_tf_224",
    # HRNet
    "hrnet_w18_small", "hrnet_w18",
    # CrossViT
    "crossvit_tiny_240", "crossvit_small_240",
    # EfficientFormer
    "efficientformer_l1", "efficientformer_l3",
    # GENet
    "gernet_s", "gernet_m",
    # VGG
    "vgg11", "vgg13", "vgg16", "vgg19",
    "vgg11_bn", "vgg13_bn", "vgg16_bn", "vgg19_bn",
    # AlexNet / SqueezeNet — historic baselines
    "alexnet", "squeezenet1_0",
]

# ── TORCHVISION ───────────────────────────────────────────────────────
TV_CANDIDATES = [
    "alexnet",
    "vgg11", "vgg13", "vgg16", "vgg19",
    "vgg11_bn", "vgg13_bn", "vgg16_bn", "vgg19_bn",
    "squeezenet1_0", "squeezenet1_1",
    "resnet18", "resnet34", "resnet50", "resnet101", "resnet152",
    "resnext50_32x4d", "wide_resnet50_2",
    "densenet121", "densenet161", "densenet169", "densenet201",
    "inception_v3", "googlenet",
    "mobilenet_v2", "mobilenet_v3_large", "mobilenet_v3_small",
    "shufflenet_v2_x0_5", "shufflenet_v2_x1_0", "shufflenet_v2_x2_0",
    "mnasnet0_5", "mnasnet1_0",
    "efficientnet_b0", "efficientnet_b1", "efficientnet_b2",
    "efficientnet_b3", "efficientnet_b4",
    "regnet_y_400mf", "regnet_y_800mf", "regnet_y_1_6gf",
    "regnet_x_400mf", "regnet_x_800mf", "regnet_x_1_6gf",
    "vit_b_16", "vit_b_32",
    "swin_t", "swin_s", "swin_b",
    "convnext_tiny", "convnext_small", "convnext_base",
]

# ── HUGGINGFACE TRANSFORMERS ──────────────────────────────────────────
# Removed: gpt2-xl (1.5 B), opt-1.3b (1.3 B), gpt-neo-1.3B, rembert (576 M),
#          deberta-v2-xlarge (900 M), bert-large / roberta-large / xlm-roberta-large
#          (each ~335 M — still heavyweight on CPU, kept only if fit)
HF_CANDIDATES = [
    # ── BERT-family ─────────────────────────────────────────────────
    ("bert-base-uncased",                      "encoder"),
    ("bert-base-cased",                        "encoder"),
    ("bert-large-uncased",                     "encoder"),   # 336 M
    ("bert-base-multilingual-uncased",         "encoder"),
    ("bert-base-multilingual-cased",           "encoder"),
    # Micro/Mini BERTs
    ("prajjwal1/bert-tiny",                    "encoder"),
    ("prajjwal1/bert-mini",                    "encoder"),
    ("prajjwal1/bert-small",                   "encoder"),
    ("prajjwal1/bert-medium",                  "encoder"),
    # ── DistilBERT ──────────────────────────────────────────────────
    ("distilbert-base-uncased",                "encoder"),
    ("distilbert-base-cased",                  "encoder"),
    ("distilbert-base-multilingual-cased",     "encoder"),
    # ── RoBERTa ─────────────────────────────────────────────────────
    ("roberta-base",                           "encoder"),
    ("xlm-roberta-base",                       "encoder"),
    # ── ALBERT ──────────────────────────────────────────────────────
    ("albert-base-v2",                         "encoder"),
    ("albert-large-v2",                        "encoder"),
    ("albert-xlarge-v2",                       "encoder"),
    # ── ELECTRA ─────────────────────────────────────────────────────
    ("google/electra-small-discriminator",     "encoder"),
    ("google/electra-small-generator",         "encoder"),
    ("google/electra-base-discriminator",      "encoder"),
    ("google/electra-base-generator",          "encoder"),
    # ── GPT-2  (≤ medium ~345 M; large/xl removed) ──────────────────
    ("gpt2",                                   "decoder"),
    ("gpt2-medium",                            "decoder"),
    ("distilgpt2",                             "decoder"),
    # ── OPT ─────────────────────────────────────────────────────────
    ("facebook/opt-125m",                      "decoder"),
    ("facebook/opt-350m",                      "decoder"),
    # ── GPT-Neo 125 M ────────────────────────────────────────────────
    ("EleutherAI/gpt-neo-125m",                "decoder"),
    # ── DeBERTa ─────────────────────────────────────────────────────
    ("microsoft/deberta-base",                 "encoder"),
    ("microsoft/deberta-v3-base",              "encoder"),
    ("microsoft/deberta-v3-small",             "encoder"),
    ("microsoft/deberta-v2-base",              "encoder"),
    # ── MobileBERT ──────────────────────────────────────────────────
    ("google/mobilebert-uncased",              "encoder"),
    # ── SqueezeBERT ─────────────────────────────────────────────────
    ("squeezebert/squeezebert-uncased",        "encoder"),
    # ── ConvBERT ────────────────────────────────────────────────────
    ("YituTech/conv-bert-base",                "encoder"),
    ("YituTech/conv-bert-small",               "encoder"),
    ("YituTech/conv-bert-medium-small",        "encoder"),
    # ── FNet ────────────────────────────────────────────────────────
    ("google/fnet-base",                       "encoder"),
    ("google/fnet-large",                      "encoder"),
    # ── MPNet ───────────────────────────────────────────────────────
    ("microsoft/mpnet-base",                   "encoder"),
    # ── CamemBERT ───────────────────────────────────────────────────
    ("camembert-base",                         "encoder"),
    # ── Funnel Transformer ──────────────────────────────────────────
    ("funnel-transformer/small-base",          "encoder"),
    ("funnel-transformer/medium-base",         "encoder"),
    # ── TinyBERT ────────────────────────────────────────────────────
    ("huawei-noah/TinyBERT_General_4L_312D",   "encoder"),
    ("huawei-noah/TinyBERT_General_6L_768D",   "encoder"),
    # ── Domain-specific BERTs ───────────────────────────────────────
    ("allenai/scibert_scivocab_uncased",       "encoder"),
    ("dmis-lab/biobert-v1.1",                  "encoder"),
    ("ProsusAI/finbert",                       "encoder"),
    ("nlpaueb/legal-bert-base-uncased",        "encoder"),
    ("microsoft/codebert-base",                "encoder"),
    # ── Sentence Transformers ───────────────────────────────────────
    ("sentence-transformers/all-MiniLM-L6-v2",        "encoder"),
    ("sentence-transformers/all-MiniLM-L12-v2",       "encoder"),
    ("sentence-transformers/paraphrase-MiniLM-L6-v2", "encoder"),
    # ── Multilingual ────────────────────────────────────────────────
    ("bert-base-german-cased",                 "encoder"),
    ("dccuchile/bert-base-spanish-wwm-cased",  "encoder"),
    ("cl-tohoku/bert-base-japanese",           "encoder"),
    # ── Tiny GPT ────────────────────────────────────────────────────
    ("sshleifer/tiny-gpt2",                    "decoder"),
    # ── OpenAI GPT-1 ────────────────────────────────────────────────
    ("openai-gpt",                             "decoder"),
    # ── XLNet ───────────────────────────────────────────────────────
    ("xlnet-base-cased",                       "decoder"),
    # ── LayoutLM ────────────────────────────────────────────────────
    ("microsoft/layoutlm-base-uncased",        "encoder"),
    # ── Longformer (sparse) ─────────────────────────────────────────
    ("allenai/longformer-base-4096",           "encoder"),
    # ── BigBird (block-sparse) ──────────────────────────────────────
    ("google/bigbird-roberta-base",            "encoder"),
    # ── CANINE (char-level) ─────────────────────────────────────────
    ("google/canine-s",                        "encoder"),
    ("google/canine-c",                        "encoder"),
]

print(f"Candidates: {len(TIMM_CANDIDATES)} timm | {len(TV_CANDIDATES)} torchvision | {len(HF_CANDIDATES)} HuggingFace")
print(f"Total: {len(TIMM_CANDIDATES)+len(TV_CANDIDATES)+len(HF_CANDIDATES)} → target 200")


Candidates: 105 timm | 51 torchvision | 63 HuggingFace
Total: 219 → target 200


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK C — RUN THE EXTRACTION SCRIPT
#
# The heavy loop runs in a child process (extract_hpc_dataset.py).
# This avoids Jupyter kernel OOM/buffer-overflow crashes.
#
# ▸ Progress is streamed to  extraction.log  in the same folder.
# ▸ Checkpoints are written every 5 successes to final_hpc_dataset.json
# ▸ Re-running is safe — already-processed models are skipped.
# ══════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time

script = os.path.join(
    os.path.dirname(os.path.abspath("models.ipynb")),
    "extract_hpc_dataset.py"
)
if not os.path.exists(script):
    script = r"c:\Users\fevzikilas\Desktop\RL\RL project\extract_hpc_dataset.py"

print(f"Launching: {script}")
print("Streaming extraction.log — this may take 1–3 hours on CPU.")
print("You can watch progress in extraction.log while this cell runs.\n")

t0 = time.time()
result = subprocess.run(
    [sys.executable, script, "--target", "200", "--save-every", "5"],
    capture_output=False,     # let stdout/stderr pass through to terminal
    text=True,
)
elapsed = time.time() - t0
print(f"\n--- Script finished in {elapsed/60:.1f} min  |  returncode={result.returncode} ---")


Candidates: 219  |  Target: 200  |  Remaining: 200
────────────────────────────────────────────────────────────
[  1/200] ✓ timm__resnet18                                           n= 173  GF=   0.01  0.1m
[  2/200] ✓ timm__resnet34                                           n= 309  GF=   0.01  0.1m
[  3/200] ✓ timm__resnet50                                           n= 444  GF=   0.04  0.2m
[  4/200] ✓ timm__resnet101                                          n= 869  GF=   0.06  0.3m
[  5/200] ✓ timm__resnet152                                          n=1294  GF=   0.08  0.6m
[  6/200] ✓ timm__wide_resnet50_2                                    n= 444  GF=   0.05  0.7m
[  7/200] ✓ timm__wide_resnet101_2                                   n= 869  GF=   0.07  0.8m
[  8/200] ✓ timm__resnext50_32x4d                                    n= 444  GF=   0.05  0.9m
[  9/200] ✓ timm__resnext101_32x8d                                   n= 869  GF=   0.10  1.1m
[ 10/200] ✓ timm__efficientnet_b0         

: 

: 

: 

In [ ]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK D — LOAD RESULTS INTO THE NOTEBOOK
# ══════════════════════════════════════════════════════════════════════
import json, os

dataset_path = r"c:\Users\fevzikilas\Desktop\RL\RL project\final_hpc_dataset.json"
failed_path  = r"c:\Users\fevzikilas\Desktop\RL\RL project\failed_models.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    final_hpc_dataset = json.load(f)

if os.path.exists(failed_path):
    with open(failed_path, "r", encoding="utf-8") as f:
        failed_models = json.load(f)
else:
    failed_models = {}

size_mb = os.path.getsize(dataset_path) / (1024 ** 2)
print(f"Loaded {len(final_hpc_dataset)} models  ({size_mb:.1f} MB)")
print(f"Failed models logged: {len(failed_models)}")


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK E — SANITY CHECK
#   1. Overall statistics (node count, FLOPs, transfer bytes)
#   2. Spot-check: one CNN, one ViT, one NLP model
#   3. Assert no feature column is all-zeros
# ══════════════════════════════════════════════════════════════════════
import random

def fmt(n):
    """Human-readable number."""
    if   n >= 1e12: return f"{n/1e12:.2f} T"
    elif n >= 1e9:  return f"{n/1e9:.2f} G"
    elif n >= 1e6:  return f"{n/1e6:.2f} M"
    elif n >= 1e3:  return f"{n/1e3:.2f} K"
    return str(n)

BAR = "─" * 72
print("═"*72)
print("  SANITY CHECK")
print("═"*72)

# ── 1. Global stats ───────────────────────────────────────────────────
all_keys = list(final_hpc_dataset.keys())
print(f"\nTotal models in dataset : {len(all_keys)}")
print(f"\n{'Key':<50} {'Nodes':>6}  {'GFLOPs':>9}  {'Trans GB':>9}")
print(BAR)
for k in all_keys:
    meta = final_hpc_dataset[k]
    print(
        f"{k[:50]:<50} {meta['node_count']:>6d}  "
        f"{meta['total_flops']/1e9:>9.3f}  "
        f"{meta['total_transfer_bytes']/1e9:>9.3f}"
    )

# ── 2. Spot-check 3 models ────────────────────────────────────────────
# Find one CNN, one ViT, one NLP representative
def find_key(keywords):
    for k in all_keys:
        kl = k.lower()
        if any(kw in kl for kw in keywords):
            return k
    return None

spot_cnn  = find_key(["resnet50", "vgg16", "densenet121", "efficientnet_b0"])
spot_vit  = find_key(["vit_base", "deit_base", "swin_base", "swin_tiny"])
spot_nlp  = find_key(["bert-base-uncased", "gpt2__", "roberta-base"])

print(f"\n{'─'*72}")
print("Spot-check (CNN / ViT / NLP):")
print(f"{'─'*72}")

for label, key in [("CNN", spot_cnn), ("ViT", spot_vit), ("NLP", spot_nlp)]:
    if key is None:
        print(f"  [{label}]  — no candidate found in dataset")
        continue

    meta   = final_hpc_dataset[key]
    dag    = meta["graph"]
    nodes  = list(dag.values())

    total_flops    = sum(n["flops"]          for n in nodes)
    total_transfer = sum(n["transfer_byte"]  for n in nodes)
    total_mem      = sum(n["mem_byte"]       for n in nodes)
    nonzero_flops  = sum(1 for n in nodes if n["flops"]          > 0)
    nonzero_trans  = sum(1 for n in nodes if n["transfer_byte"]  > 0)

    ok_flops = total_flops    > 0
    ok_trans = total_transfer > 0

    print(f"\n  [{label}]  {key}")
    print(f"         nodes           : {len(nodes)}")
    print(f"         total FLOPs     : {fmt(total_flops)}  "
          f"({'✓' if ok_flops else '✗ ZERO!'})")
    print(f"         total TransByte : {fmt(total_transfer)}  "
          f"({'✓' if ok_trans else '✗ ZERO!'})")
    print(f"         total MemByte   : {fmt(total_mem)}")
    print(f"         nodes w/ FLOPs>0: {nonzero_flops}/{len(nodes)}")
    print(f"         nodes w/ Trans>0: {nonzero_trans}/{len(nodes)}")

    # Assert
    assert ok_flops, f"FLOPs are all zero for {key}!"
    assert ok_trans, f"transfer_byte is all zero for {key}!"

# ── 3. Per-source breakdown ───────────────────────────────────────────
print(f"\n{'─'*72}")
print("Per-source breakdown:")
print(f"{'─'*72}")
from collections import Counter
src_counts = Counter(v["source"] for v in final_hpc_dataset.values())
for src, cnt in sorted(src_counts.items()):
    print(f"  {src:<14s} : {cnt:>3d} models")

print(f"\n{'═'*72}")
print("  All checks passed ✓" if success_count > 0 else "  No models were extracted!")
print(f"{'═'*72}")


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# BLOCK F — BUILD TWO-SPLIT DATASET & PUSH TO HUggingFace Hub
#
#   Split 1 — "models"  : one row per model  (metadata + aggregate stats)
#   Split 2 — "nodes"   : one row per graph node  (HPC features per op)
# ══════════════════════════════════════════════════════════════════════
import json, os, io
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, Sequence
from huggingface_hub import DatasetCard

# ── 1. Load checkpoint ────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    print("Google Colab algılandı.")
    print("Lütfen 'final_hpc_dataset.json' dosyasını yükleyin...")
    uploaded = colab_files.upload()           # kullanıcıya dosya seçtir
    first_key = list(uploaded.keys())[0]
    data = json.load(io.BytesIO(uploaded[first_key]))
    print(f"✓ '{first_key}' yüklendi")
except ImportError:
    # Colab değil — yerel Windows yolu
    dataset_path = r"c:\Users\fevzikilas\Desktop\RL\RL project\final_hpc_dataset.json"
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"✓ Yerel dosya yüklendi: {dataset_path}")

print(f"Loaded {len(data)} models from checkpoint")

# ── 2. Build rows ─────────────────────────────────────────────────────
models_rows = []
nodes_rows  = []

for m_id, content in data.items():
    # ── models split ──────────────────────────────────────────────────
    models_rows.append({
        "model_id":             m_id,
        "source":               content["source"],
        "model_name":           content["model_name"],
        "export_mode":          content.get("export_mode", "strict"),
        "node_count":           int(content["node_count"]),
        "total_flops":          float(content["total_flops"]),
        "total_transfer_bytes": float(content["total_transfer_bytes"]),
    })

    # ── nodes split ───────────────────────────────────────────────────
    for idx, (node_name, feat) in enumerate(content["graph"].items()):
        nodes_rows.append({
            "model_id":     m_id,
            "source":       content["source"],
            "model_name":   content["model_name"],
            "node_index":   idx,
            "node_name":    node_name,
            "flops":        float(feat["flops"]),
            "mem_byte":     float(feat["mem_byte"]),
            "transfer_byte":float(feat["transfer_byte"]),
            "parents":      feat["parents"],          # list[str]
        })

avg_nodes = len(nodes_rows) / max(len(models_rows), 1)
print(f"models split : {len(models_rows):,} rows")
print(f"nodes  split : {len(nodes_rows):,} rows  (avg {avg_nodes:.0f} nodes/model)")

# ── 3. Explicit Feature schema (proper Arrow types) ───────────────────
models_features = Features({
    "model_id":             Value("string"),
    "source":               Value("string"),
    "model_name":           Value("string"),
    "export_mode":          Value("string"),
    "node_count":           Value("int32"),
    "total_flops":          Value("float64"),
    "total_transfer_bytes": Value("float64"),
})

nodes_features = Features({
    "model_id":     Value("string"),
    "source":       Value("string"),
    "model_name":   Value("string"),
    "node_index":   Value("int32"),
    "node_name":    Value("string"),
    "flops":        Value("float64"),
    "mem_byte":     Value("float64"),
    "transfer_byte":Value("float64"),
    "parents":      Sequence(Value("string")),
})

# ── 4. Build DatasetDict ──────────────────────────────────────────────
models_ds = Dataset.from_list(models_rows, features=models_features)
nodes_ds  = Dataset.from_list(nodes_rows,  features=nodes_features)

dataset_dict = DatasetDict({
    "models": models_ds,
    "nodes":  nodes_ds,
})

print("\n", dataset_dict)

# ── 5. Quick preview — first model's nodes as a table ─────────────────
first_id = models_rows[0]["model_id"]
preview_df = (
    nodes_ds
    .filter(lambda x: x["model_id"] == first_id)
    .to_pandas()
    [["node_name", "flops", "mem_byte", "transfer_byte"]]
)
print(f"\nPreview — nodes of '{first_id}' (top 15 by flops):")
print(preview_df.nlargest(15, "flops").to_string(index=False))

# ── 6. Dataset Card (README) ──────────────────────────────────────────
CARD = """\
---
license: apache-2.0
task_categories:
  - graph-ml
  - other
language:
  - en
tags:
  - deep-learning
  - computational-graphs
  - dag
  - hpc
  - flops
  - neural-architecture
  - pytorch
  - timm
  - torchvision
  - transformers
pretty_name: DL Architectural DAGs — HPC Features
size_categories:
  - 1M<n<10M
---

# DL Architectural DAGs — HPC Features

A research dataset of **computation graphs (DAGs)** extracted from 200+ deep learning models
via `torch.export`, annotated with hardware-performance-counter (HPC) proxy features per node.

Designed for research on **operator scheduling, memory mapping, and performance prediction**
of DNN workloads on heterogeneous hardware (CPUs, GPUs, accelerators).

---

## Splits

| Split | Rows | Description |
|-------|-----:|-------------|
| `models` | ~200 | One row per model — metadata + aggregate stats |
| `nodes` | ~300 000 | One row per graph node — per-operation HPC features |

---

## Schema

### `models` split

| Column | Type | Description |
|--------|------|-------------|
| `model_id` | string | Unique key: `{source}__{model_name}` |
| `source` | string | `timm` / `torchvision` / `hf` |
| `model_name` | string | Original model identifier |
| `export_mode` | string | `strict` or `non-strict` (`torch.export` mode used) |
| `node_count` | int32 | Total number of graph nodes |
| `total_flops` | float64 | Sum of FLOPs across all nodes |
| `total_transfer_bytes` | float64 | Sum of output-tensor bytes across all nodes |

### `nodes` split

| Column | Type | Description |
|--------|------|-------------|
| `model_id` | string | Foreign key → `models.model_id` |
| `source` | string | `timm` / `torchvision` / `hf` |
| `model_name` | string | Model this node belongs to |
| `node_index` | int32 | Topological order index in the graph |
| `node_name` | string | ATen IR node name (e.g. `addmm`, `convolution`) |
| `flops` | float64 | Estimated FLOPs (MACs × 2) |
| `mem_byte` | float64 | Input-tensor bytes read by this node (weight-load proxy) |
| `transfer_byte` | float64 | Output-tensor bytes produced (data-volume / communication proxy) |
| `parents` | list[string] | Predecessor node names — encodes DAG edges |

---

## Models Covered

| Library | Count | Architectures |
|---------|------:|---------------|
| **timm** | ~100 | ResNet, EfficientNet, ViT, DeiT, Swin, ConvNeXt, MobileNet, DenseNet, RegNet, MLP-Mixer, ResMLP, CaiT, LeViT, NFNet, Twins, PVT, HRNet, CrossViT, EfficientFormer, VGG, … |
| **torchvision** | ~51 | AlexNet, VGG, SqueezeNet, ResNet, DenseNet, Inception, GoogLeNet, MobileNet, ShuffleNet, EfficientNet, RegNet, ViT-B, Swin-T/S/B, ConvNeXt, … |
| **HuggingFace Transformers** | ~50 | BERT family, DistilBERT, RoBERTa, ALBERT, ELECTRA, GPT-2, OPT, GPT-Neo, DeBERTa, MobileBERT, FNet, MPNet, TinyBERT, domain-BERTs, … |

---

## FLOPs Estimation Method

| Operation | Formula |
|-----------|---------|
| `mm` / `addmm` (Linear) | `2 × M × N × K` |
| `bmm` (batched matmul) | `2 × B × M × N × K` |
| `convolution` (any-D, grouped) | `2 × output_elements × (C_in/g × ∏ kernel_dims)` |
| `scaled_dot_product_attention` | `2 × B × H × L² × D` |
| `einsum` (DeBERTa disentangled) | `2 × output_elements` |
| element-wise (ReLU, Add, LayerNorm, …) | `output_elements` |

---

## Usage

```python
from datasets import load_dataset
import pandas as pd

ds = load_dataset("nieche/DL-Architectural-DAGs-2026")

# ── All nodes of ResNet-50 from timm
resnet_nodes = ds["nodes"].filter(lambda x: x["model_id"] == "timm__resnet50")

# ── Top-10 most FLOPs-intensive nodes across the whole dataset
df = ds["nodes"].to_pandas()
print(df.nlargest(10, "flops")[["model_id", "node_name", "flops"]])

# ── Aggregate FLOPs per model family
models_df = ds["models"].to_pandas()
print(models_df.groupby("source")["total_flops"].describe())

# ── Build adjacency list for a graph-learning framework
import networkx as nx
g = nx.DiGraph()
for row in resnet_nodes:
    for parent in row["parents"]:
        g.add_edge(parent, row["node_name"],
                   flops=row["flops"], transfer=row["transfer_byte"])
```

---

## Citation

```bibtex
@dataset{dl_architectural_dags_2026,
  title  = {DL Architectural DAGs — HPC Features},
  year   = {2026},
  url    = {https://huggingface.co/datasets/nieche/DL-Architectural-DAGs-2026}
}
```
"""


Google Colab algılandı.
Lütfen 'final_hpc_dataset.json' dosyasını yükleyin...


: 

In [10]:

dataset_dict

DatasetDict({
    models: Dataset({
        features: ['model_id', 'source', 'model_name', 'export_mode', 'node_count', 'total_flops', 'total_transfer_bytes'],
        num_rows: 70
    })
    nodes: Dataset({
        features: ['model_id', 'source', 'model_name', 'node_index', 'node_name', 'flops', 'mem_byte', 'transfer_byte', 'parents'],
        num_rows: 51200
    })
})

In [ ]:

# ── 7. HuggingFace Login & Push to Hub ────────────────────────────────

from huggingface_hub import login, DatasetCard

HF_TOKEN = "xxx"  
HF_REPO  = "nieche/DL-Architectural-DAGs-2026"

login(token=HF_TOKEN)

dataset_dict.push_to_hub(HF_REPO, private=False)

card = DatasetCard(CARD)
card.push_to_hub(HF_REPO)

print(f"\n✓ Dataset pushed → https://huggingface.co/datasets/{HF_REPO}")
